# 02 — Audit the source xlsx as the graph-sourcing candidate

Follow-up to [`01_verify_graph_walk.ipynb`](01_verify_graph_walk.ipynb).

**Why this notebook exists.** Notebook 01 showed that `yaml/20260331_r16/sdtm/` is a partial publication — 227 DSSs across 5 domains only. The xlsx at `cdisc-org.github.io/COSMoS/export/cdisc_sdtm_dataset_specializations_latest.xlsx` is what the current pipeline has been using since inception. Before deciding Step 2 sourcing, we need to be evidence-driven about what that xlsx actually contains.

**Hypothesis this notebook tests.** The source xlsx carries the full authored graph at VLM-row grain (one row per `SDTMVariable` node in LinkML terms), with column names that map 1:1 to LinkML slot names. If true, Step 2 can use the xlsx as sourcing and still be a graph-walk flattener — the walk is over pandas rows grouped by DSS, not over YAML files.

**Contrast with the Step 1 doc framing.** `COSMoS_Graph_As_Authored.md` §7 describes the xlsx as "the lossiest of the available serialisations". This notebook tests whether that framing conflates the *source* xlsx (authored graph export) with the *interim* `COSMoS_BC_DSS.xlsx` (our derivation that collapses VLM rows). The latter is lossy by design; the former needs measurement.

**Scope.**

1. Load the three-sheet xlsx; cover the ReadMe stated total.
2. Column-by-column population counts.
3. Domain and DSS coverage — compare to the YAML folder's 5 domains / 227 DSSs.
4. `vlm_source` distribution — distinct pinning patterns.
5. Reification coverage at row grain — RelationShip quad, assigned_term, codelist/subset/value_list, origin_type/origin_source.
6. Structural equivalence check: ABI from the xlsx vs the cached `sdtm_abi.yaml` — do they carry the same facts?
7. Column → LinkML slot mapping via `SchemaView`.
8. Presence check on the case-pair DSSs that motivated the rework.
9. Findings summary supporting a sourcing decision.


## 1. Imports


In [1]:
import sys
from pathlib import Path
from collections import Counter

import yaml
import pandas as pd

from linkml_runtime.utils.schemaview import SchemaView

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
print('python :', sys.version.split()[0])
print('pandas :', pd.__version__)


python : 3.12.6
pandas : 3.0.1


## 2. Configuration

The xlsx is the current pipeline's source, already cached at `cosmos-bc-dss/downloads/cdisc_sdtm_dataset_specializations_latest.xlsx`. The LinkML schema and the ABI sample YAML are at `reference/cosmos_linkml/` (gitignored local cache).


In [2]:
REPO_ROOT = Path.cwd().parent.parent  # notebooks/ -> cosmos-bc-dss/ -> repo root

XLSX_PATH = Path('..') / 'downloads' / 'cdisc_sdtm_dataset_specializations_latest.xlsx'
SCHEMA_DIR = REPO_ROOT / 'reference' / 'cosmos_linkml'
SDTM_SCHEMA = SCHEMA_DIR / 'cosmos_sdtm_model.yaml'
ABI_YAML = SCHEMA_DIR / 'sdtm_abi.yaml'

assert XLSX_PATH.exists(), f'xlsx not found at {XLSX_PATH.resolve()} — run the existing Flatten notebook first to download it'
assert SDTM_SCHEMA.exists(), f'schema not found at {SDTM_SCHEMA}'
assert ABI_YAML.exists(), f'ABI sample not found at {ABI_YAML}'

print('xlsx   :', XLSX_PATH.resolve())
print('schema :', SDTM_SCHEMA)
print('yaml   :', ABI_YAML)


xlsx   : /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/cosmos-bc-dss/downloads/cdisc_sdtm_dataset_specializations_latest.xlsx
schema : /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/reference/cosmos_linkml/cosmos_sdtm_model.yaml
yaml   : /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/reference/cosmos_linkml/sdtm_abi.yaml


## 3. Sheet structure and ReadMe claim

The xlsx has three sheets: a human-readable ReadMe with the authoritative package count, a wide data sheet with one row per VLM variable, and a Domains enumeration.


In [3]:
xl = pd.ExcelFile(XLSX_PATH)
print('sheets:', xl.sheet_names)

# ReadMe carries the authoritative 'unique DSS count' for the package
readme = pd.read_excel(XLSX_PATH, sheet_name='ReadMe', header=None)
print()
print('ReadMe content:')
print(readme.iloc[0, 0])


sheets: ['ReadMe', 'SDTM Dataset Specializations', 'Domains']

ReadMe content:
This spreadsheet contains the latest versions of CDISC SDTM Dataset Specializations in the CDISC Library as of 2026-03-31.
There are currently 1326 unique CDISC SDTM Dataset Specializations in the CDISC Library.
The image on the right shows the relation between Biomedical Concepts and SDTM Dataset Specializations.
Only a few attributes are shown in the image.


In [4]:
df = pd.read_excel(XLSX_PATH, sheet_name='SDTM Dataset Specializations')
print(f'rows : {len(df):,}')
print(f'cols : {len(df.columns)}')
print()
print('columns:')
for c in df.columns:
    print('  ', c)


rows : 12,677
cols : 32

columns:
   package_date
   bc_id
   sdtmig_start_version
   sdtmig_end_version
   domain
   vlm_source
   vlm_group_id
   short_name
   sdtm_variable
   dec_id
   nsv_flag
   codelist
   codelist_submission_value
   subset_codelist
   value_list
   assigned_term
   assigned_value
   role
   subject
   linking_phrase
   predicate_term
   object
   data_type
   length
   format
   significant_digits
   mandatory_variable
   mandatory_value
   origin_type
   origin_source
   comparator
   vlm_target


In [5]:
domains_sheet = pd.read_excel(XLSX_PATH, sheet_name='Domains')
print(f'Domains sheet: {len(domains_sheet)} rows')
print(domains_sheet.to_string(index=False))


Domains sheet: 32 rows
domain
    AE
    BE
    CM
    DD
    DM
    DS
    EC
    EG
    EX
    FA
    FT
    GF
    IE
    IS
    LB
    MB
    MH
    MI
    MK
    PR
    QS
    RE
    RP
    RS
    SC
    SR
    SU
    TR
    TS
    TU
    UR
    VS


## 4. Column population counts

How full is each column? A column that is sparsely populated is not necessarily missing information — some columns (e.g. `assigned_term`, `comparator`) are only meaningful when a DSS pins a value, and sparse population is semantically correct.


In [6]:
pop = pd.DataFrame({
    'column':   df.columns,
    'non_null': [int(df[c].notna().sum()) for c in df.columns],
})
pop['fill_pct'] = (pop['non_null'] / len(df) * 100).round(1)
pop = pop.sort_values('non_null', ascending=False).reset_index(drop=True)
pop


,column,non_null,fill_pct
0,package_date,12677,100.0
1,sdtmig_start_version,12677,100.0
2,mandatory_value,12677,100.0
3,domain,12677,100.0
4,vlm_source,12677,100.0
5,vlm_group_id,12677,100.0
6,short_name,12677,100.0
7,sdtm_variable,12677,100.0
8,nsv_flag,12677,100.0
9,bc_id,12677,100.0


## 5. Domain and DSS coverage

How many distinct domains and DSSs does the xlsx cover? How does that compare to the YAML folder's 5 domains / 227 DSSs found in 01?


In [7]:
n_dss = df['vlm_group_id'].nunique()
n_domains = df['domain'].nunique()
print(f'distinct DSSs   : {n_dss}')
print(f'distinct domains: {n_domains}')
print()
print('DSSs per domain:')
print(df.groupby('domain')['vlm_group_id'].nunique().sort_values(ascending=False).to_string())


distinct DSSs   : 1326
distinct domains: 32

DSSs per domain:
domain
IS    290
LB    146
RS    135
RE    135
TS    129
RP     96
VS     78
MK     50
GF     38
EG     33
FT     23
DS     20
QS     17
DD     13
FA     12
SR     12
SU     12
MH     11
TU     10
UR     10
TR      9
PR      8
MI      7
BE      7
MB      7
DM      5
CM      4
IE      2
SC      2
EC      2
AE      2
EX      1


In [8]:
# Coverage gap vs yaml/20260331_r16/sdtm/
YAML_DOMAINS = {'RE', 'VS', 'DS', 'LB', 'GF'}
xlsx_domains = set(df['domain'].dropna().unique())
missing_from_yaml = sorted(xlsx_domains - YAML_DOMAINS)
missing_dss_count = df[df['domain'].isin(missing_from_yaml)]['vlm_group_id'].nunique()

print(f'domains in xlsx : {len(xlsx_domains)}')
print(f'domains in yaml : {len(YAML_DOMAINS)}')
print(f'missing from yaml: {len(missing_from_yaml)} domains, {missing_dss_count} DSSs')
print(f'  -> {missing_from_yaml}')


domains in xlsx : 32
domains in yaml : 5
missing from yaml: 27 domains, 909 DSSs
  -> ['AE', 'BE', 'CM', 'DD', 'DM', 'EC', 'EG', 'EX', 'FA', 'FT', 'IE', 'IS', 'MB', 'MH', 'MI', 'MK', 'PR', 'QS', 'RP', 'RS', 'SC', 'SR', 'SU', 'TR', 'TS', 'TU', 'UR']


## 6. `vlm_source` distribution

Notebook 01 found 5 distinct `source` values in the YAML folder. How many distinct `vlm_source` values are in the xlsx? Each one is a `<domain>.<variable>` pinning pattern. This number sets the lower bound on how many DSS shapes the Step 2 flattener must accommodate.


In [9]:
# One source per DSS (take one row per vlm_group_id)
dss_source = df.groupby('vlm_group_id')['vlm_source'].first()
source_counts = dss_source.value_counts(dropna=False)
print(f'distinct vlm_source values across {len(dss_source)} DSSs: {dss_source.nunique(dropna=False)}')
print()
print(source_counts.to_string())


distinct vlm_source values across 1326 DSSs: 37

vlm_source
IS.ISTESTCD    290
LB.LBTESTCD    146
RE.RETESTCD    135
RS.RSTESTCD    135
TS.TSPARMCD    129
RP.RPTESTCD     96
VS.VSTESTCD     78
MK.MKTESTCD     50
GF.GFTESTCD     38
EG.EGTESTCD     33
FT.FTTESTCD     23
DS.DECODE       20
QS.QSTESTCD     17
DD.DDTESTCD     13
FA.FATESTCD     12
SU.SUTRT        12
SR.SRTESTCD     12
MH.MHTERM       11
UR.URTESTCD     10
TU.TUTESTCD     10
TR.TRTESTCD      9
PR.PRTRT         8
BE.BETERM        7
MI.MITESTCD      7
CM.CMTRT         4
MB-MBTESTCD      4
MB.MBTESTCD      3
AE.AETERM        2
SC.SCTESTCD      2
IE.IETESTCD      2
EC.ECTRT         2
DM.AGE           1
DM.BRTHDTC       1
DM.ETHNIC        1
EX.EXTRT         1
DM.RACE          1
DM.SEX           1


## 7. Reification coverage at row grain

Does the xlsx carry the `RelationShip` quad (`subject / linking_phrase / predicate_term / object`), the `AssignedTerm` pair (`assigned_term / assigned_value`), `codelist`, `subset_codelist`, `value_list`, `origin_type`, `origin_source`? If yes, the xlsx is structurally equivalent to the YAML at VLM-row grain.


In [10]:
reification_cols = [
    'subject', 'linking_phrase', 'predicate_term', 'object',
    'assigned_term', 'assigned_value',
    'codelist', 'codelist_submission_value', 'subset_codelist', 'value_list',
    'role', 'mandatory_variable', 'mandatory_value', 'comparator',
    'origin_type', 'origin_source',
]
missing = [c for c in reification_cols if c not in df.columns]
print('reification columns missing from xlsx:', missing or '(none)')

present = [c for c in reification_cols if c in df.columns]
coverage = pd.DataFrame({
    'column':         present,
    'rows_populated': [int(df[c].notna().sum()) for c in present],
    'fill_pct':       [round(df[c].notna().mean() * 100, 1) for c in present],
})
coverage


reification columns missing from xlsx: (none)


,column,rows_populated,fill_pct
0,subject,12364,97.5
1,linking_phrase,12364,97.5
2,predicate_term,12364,97.5
3,object,12364,97.5
4,assigned_term,3994,31.5
5,assigned_value,4478,35.3
6,codelist,6847,54.0
7,codelist_submission_value,6847,54.0
8,subset_codelist,279,2.2
9,value_list,2528,19.9


In [11]:
# DSS-level coverage — how many DSSs have at least one populated row in each column?
dss_presence = {}
for c in present:
    dss_with = df.groupby('vlm_group_id')[c].apply(lambda s: s.notna().any()).sum()
    dss_presence[c] = int(dss_with)
dss_presence_df = pd.DataFrame({
    'column': list(dss_presence),
    'dss_with_any_value': list(dss_presence.values()),
})
dss_presence_df['pct_of_dss'] = (dss_presence_df['dss_with_any_value'] / n_dss * 100).round(1)
dss_presence_df.sort_values('dss_with_any_value', ascending=False).reset_index(drop=True)


,column,dss_with_any_value,pct_of_dss
0,role,1326,100.0
1,mandatory_variable,1326,100.0
2,mandatory_value,1326,100.0
3,codelist,1325,99.9
4,codelist_submission_value,1325,99.9
5,subject,1322,99.7
6,linking_phrase,1322,99.7
7,predicate_term,1322,99.7
8,object,1322,99.7
9,assigned_value,1313,99.0


## 8. Structural equivalence check — ABI xlsx vs ABI YAML

The same DSS (ABI) appears in both sources. Same six variables, same pins, same relationships? If the fact sets match exactly, the xlsx is a lossless serialisation of the YAML content at VLM-row grain.


In [12]:
abi_rows = df[df['vlm_group_id'] == 'ABI'].reset_index(drop=True)
print(f'ABI rows in xlsx: {len(abi_rows)}')
abi_rows[['sdtm_variable','role','assigned_term','assigned_value','codelist',
          'subject','linking_phrase','predicate_term','object',
          'mandatory_variable','mandatory_value','comparator',
          'origin_type','origin_source']]


ABI rows in xlsx: 6


,sdtm_variable,role,assigned_term,assigned_value,codelist,subject,linking_phrase,predicate_term,object,mandatory_variable,mandatory_value,comparator,origin_type,origin_source
0,VSTESTCD,Topic,C87304,ABI,C66741,VSTESTCD,is the code for the value in,IS_DECODED_BY,VSTEST,Y,Y,EQ,NaN,NaN
1,VSTEST,Qualifier,C87304,Ankle-Brachial Index,C67153,VSTEST,decodes the value in,DECODES,VSTESTCD,Y,Y,NaN,NaN,NaN
2,VSORRES,Qualifier,NaN,NaN,NaN,VSORRES,is the result of the test in,IS_RESULT_OF,VSTESTCD,Y,N,NaN,Collected,Investigator
3,VSSTRESC,Qualifier,NaN,NaN,NaN,VSSTRESC,is the result of the test in,IS_RESULT_OF,VSTESTCD,N,N,NaN,NaN,NaN
4,VSSTRESN,Qualifier,NaN,NaN,NaN,VSSTRESN,is the result of the test in,IS_RESULT_OF,VSTESTCD,N,N,NaN,NaN,NaN
5,VSDTC,Timing,NaN,NaN,NaN,VSDTC,is the date of occurrence for,IS_TIMING_FOR,VSTESTCD,Y,N,NaN,NaN,NaN


In [13]:
abi_yaml = yaml.safe_load(ABI_YAML.read_text())
print(f'ABI variables in YAML: {len(abi_yaml["variables"])}')

# Build a per-variable comparison: xlsx row vs yaml variable dict
import json as _json

def xlsx_row_to_facts(r):
    out = {}
    for k, v in r.items():
        if pd.isna(v):
            continue
        out[k] = v
    return out

def yaml_var_to_facts(v):
    # Flatten yaml variable into the same key names the xlsx uses
    flat = {}
    flat['sdtm_variable'] = v.get('name')
    flat['role'] = v.get('role')
    if v.get('assignedTerm'):
        flat['assigned_term']  = v['assignedTerm'].get('conceptId')
        flat['assigned_value'] = v['assignedTerm'].get('value')
    if v.get('codelist'):
        flat['codelist'] = v['codelist'].get('conceptId')
        flat['codelist_submission_value'] = v['codelist'].get('submissionValue')
    if v.get('relationship'):
        r = v['relationship']
        flat['subject']        = r.get('subject')
        flat['linking_phrase'] = r.get('linkingPhrase')
        flat['predicate_term'] = r.get('predicateTerm')
        flat['object']         = r.get('object')
    if 'mandatoryVariable' in v:
        flat['mandatory_variable'] = 'Y' if v['mandatoryVariable'] else 'N'
    if 'mandatoryValue' in v:
        flat['mandatory_value'] = 'Y' if v['mandatoryValue'] else 'N'
    if 'comparator' in v:
        flat['comparator'] = v['comparator']
    if 'originType' in v:
        flat['origin_type'] = v['originType']
    if 'originSource' in v:
        flat['origin_source'] = v['originSource']
    return flat

by_var_xlsx = {r['sdtm_variable']: xlsx_row_to_facts(r) for _, r in abi_rows.iterrows()}
by_var_yaml = {v['name']: yaml_var_to_facts(v)        for v in abi_yaml['variables']}

print()
print('variables in xlsx :', sorted(by_var_xlsx))
print('variables in yaml :', sorted(by_var_yaml))

# Compare each variable's fact set
print()
print('per-variable comparison:')
for name in sorted(set(by_var_xlsx) | set(by_var_yaml)):
    x = by_var_xlsx.get(name, {})
    y = by_var_yaml.get(name, {})
    keys = set(x) | set(y)
    mismatches = []
    for k in sorted(keys):
        if k in ('vlm_group_id','bc_id','domain','package_date','sdtmig_start_version',
                 'sdtmig_end_version','vlm_source','short_name','vlm_target',
                 'data_type','length','format','significant_digits',
                 'dec_id','nsv_flag'):
            # columns only in xlsx envelope, not per-variable in the yaml shape at this level
            continue
        if x.get(k) != y.get(k):
            mismatches.append((k, x.get(k), y.get(k)))
    print(f'  {name:10s} -> {"match" if not mismatches else f"{len(mismatches)} diff(s)"}')
    for k, xv, yv in mismatches:
        print(f'        {k:24s} xlsx={xv!r:30s} yaml={yv!r}')


ABI variables in YAML: 6

variables in xlsx : ['VSDTC', 'VSORRES', 'VSSTRESC', 'VSSTRESN', 'VSTEST', 'VSTESTCD']
variables in yaml : ['VSDTC', 'VSORRES', 'VSSTRESC', 'VSSTRESN', 'VSTEST', 'VSTESTCD']

per-variable comparison:
  VSDTC      -> match
  VSORRES    -> match
  VSSTRESC   -> match
  VSSTRESN   -> match
  VSTEST     -> match
  VSTESTCD   -> match


## 9. Column → LinkML slot mapping via `SchemaView`

Confirm each xlsx column corresponds to a slot in the LinkML schema (either snake_case matches camelCase after normalisation, or the slot exists under a slightly different name). Any column that does not resolve is either a pipeline-introduced artefact or a schema gap worth flagging.


In [14]:
sv = SchemaView(str(SDTM_SCHEMA))
all_slots = {s.lower().replace('_','') for s in sv.all_slots()}

# Rename exceptions: xlsx uses 'vlm_group_id' for 'datasetSpecializationId',
# 'vlm_source' for 'source', 'bc_id' for 'biomedicalConceptId', 'short_name' for 'shortName',
# 'sdtm_variable' for 'name' (inside SDTMVariable), 'nsv_flag' for 'isNonStandard',
# 'dec_id' for 'dataElementConceptId', 'codelist_submission_value' from the inlined CodeList.submissionValue,
# 'assigned_term/value' from AssignedTerm.conceptId/value,
# 'subject/linking_phrase/predicate_term/object' from RelationShip slots.
KNOWN_RENAMES = {
    'vlm_group_id':              'datasetSpecializationId (SDTMGroup)',
    'vlm_source':                'source (SDTMGroup)',
    'bc_id':                     'biomedicalConceptId (SDTMGroup)',
    'short_name':                'shortName (SDTMGroup)',
    'sdtmig_start_version':      'sdtmigStartVersion (SDTMGroup)',
    'sdtmig_end_version':        'sdtmigEndVersion (SDTMGroup)',
    'package_date':              'packageDate (SDTMGroup)',
    'sdtm_variable':             'name (SDTMVariable)',
    'nsv_flag':                  'isNonStandard (SDTMVariable)',
    'dec_id':                    'dataElementConceptId (SDTMVariable)',
    'codelist':                  'codelist.conceptId (CodeList)',
    'codelist_submission_value': 'codelist.submissionValue (CodeList)',
    'assigned_term':             'assignedTerm.conceptId (AssignedTerm)',
    'assigned_value':            'assignedTerm.value (AssignedTerm)',
    'subject':                   'relationship.subject (RelationShip)',
    'linking_phrase':            'relationship.linkingPhrase (RelationShip)',
    'predicate_term':            'relationship.predicateTerm (RelationShip)',
    'object':                    'relationship.object (RelationShip)',
}

mapping = []
for col in df.columns:
    key = col.lower().replace('_','')
    if col in KNOWN_RENAMES:
        slot = KNOWN_RENAMES[col]
        status = 'renamed'
    elif key in all_slots:
        # Return canonical slot name from schema
        canon = next(s for s in sv.all_slots() if s.lower().replace('_','') == key)
        slot = canon
        status = 'direct'
    else:
        slot = None
        status = 'UNMAPPED'
    mapping.append({'xlsx_column': col, 'linkml_slot': slot, 'status': status})

map_df = pd.DataFrame(mapping)
map_df


,xlsx_column,linkml_slot,status
0,package_date,packageDate (SDTMGroup),renamed
1,bc_id,biomedicalConceptId (SDTMGroup),renamed
2,sdtmig_start_version,sdtmigStartVersion (SDTMGroup),renamed
3,sdtmig_end_version,sdtmigEndVersion (SDTMGroup),renamed
4,domain,domain,direct
5,vlm_source,source (SDTMGroup),renamed
6,vlm_group_id,datasetSpecializationId (SDTMGroup),renamed
7,short_name,shortName (SDTMGroup),renamed
8,sdtm_variable,name (SDTMVariable),renamed
9,dec_id,dataElementConceptId (SDTMVariable),renamed


In [15]:
unmapped = map_df[map_df['status'] == 'UNMAPPED']
print(f'unmapped columns: {len(unmapped)}')
if len(unmapped):
    print(unmapped.to_string(index=False))


unmapped columns: 0


## 10. Case-pair DSS presence check

The DSSs referenced by our three case pairs — Glucose (LB), 6MWT (QS/FT), X-Ray (PR/MK). Confirm each is present in the xlsx with the expected row count.


In [16]:
CASE_PAIR_DSS = [
    # Glucose (LB)
    'GLUCSER', 'GLUCPLS', 'GLUCRBC', 'GLUCWBL', 'GLUCUR',
    # X-Ray (PR and MK)
    'XRAY', 'XRAYCHEST', 'CTSCAN', 'CTSCANCHEST', 'MRI', 'MRIBRAIN',
    'RADIATIONCANCER', 'RADTHERAPHYBREASTCANCER',
    'SGBESCR', 'SVDHESCR',
    # 6MWT family (FT or QS)
    'SIXMWT', 'TENMW', 'TENMW101',
]
rows = []
for dss in CASE_PAIR_DSS:
    sub = df[df['vlm_group_id'] == dss]
    dom = sub['domain'].iloc[0] if len(sub) else None
    src = sub['vlm_source'].iloc[0] if len(sub) else None
    rows.append({'DS_Code': dss, 'xlsx_rows': len(sub), 'domain': dom, 'vlm_source': src})
pd.DataFrame(rows)


,DS_Code,xlsx_rows,domain,vlm_source
0,GLUCSER,12,LB,LB.LBTESTCD
1,GLUCPLS,0,NaN,NaN
2,GLUCRBC,0,NaN,NaN
3,GLUCWBL,0,NaN,NaN
4,GLUCUR,0,NaN,NaN
5,XRAY,9,PR,PR.PRTRT
6,XRAYCHEST,9,PR,PR.PRTRT
7,CTSCAN,9,PR,PR.PRTRT
8,CTSCANCHEST,9,PR,PR.PRTRT
9,MRI,9,PR,PR.PRTRT


## 11. Findings summary — sourcing decision evidence

The cell below renders a compact summary. If it reports (a) xlsx domain count > YAML's 5, (b) RelationShip quad populated on most rows, (c) ABI xlsx-vs-YAML match, (d) no unmapped columns, and (e) all case-pair DSSs present, then the xlsx is confirmed as the full graph source at VLM-row grain and Step 2 can be built on pandas-over-xlsx rather than YAML walking.


In [17]:
print('=== xlsx source audit — 2026-Q1 ===')
print(f'  xlsx path                        : {XLSX_PATH.resolve().name}')
print(f'  rows                             : {len(df):,}')
print(f'  distinct DSSs                    : {n_dss:,}')
print(f'  distinct domains                 : {n_domains}')
print(f'  distinct vlm_source patterns     : {dss_source.nunique(dropna=False)}')
print(f'  YAML coverage (from notebook 01) : 227 DSSs, 5 domains')
print(f'  xlsx-only DSSs (not in YAML)     : {missing_dss_count:,}')
print(f'  reification columns missing      : {len([c for c in reification_cols if c not in df.columns])}')
print(f'  unmapped xlsx columns vs LinkML  : {int((map_df["status"]=="UNMAPPED").sum())}')
print(f'  case-pair DSSs present           : {sum(1 for d in CASE_PAIR_DSS if (df["vlm_group_id"]==d).any())}/{len(CASE_PAIR_DSS)}')


=== xlsx source audit — 2026-Q1 ===
  xlsx path                        : cdisc_sdtm_dataset_specializations_latest.xlsx
  rows                             : 12,677
  distinct DSSs                    : 1,326
  distinct domains                 : 32
  distinct vlm_source patterns     : 37
  YAML coverage (from notebook 01) : 227 DSSs, 5 domains
  xlsx-only DSSs (not in YAML)     : 909
  reification columns missing      : 0
  unmapped xlsx columns vs LinkML  : 0
  case-pair DSSs present           : 10/18
